In [1]:
import geopandas as gpd
import pandas as pd

# Load all three layers
precincts = gpd.read_file('../data/processed/travis_precincts_2020.gpkg')
districts = gpd.read_file('../data/processed/travis_districts_2026.gpkg')
blocks = gpd.read_file('../data/processed/travis_blocks_2020.gpkg')

print(f"Precincts: {len(precincts)} | CRS: {precincts.crs.to_epsg()}")
print(f"Districts: {len(districts)} | CRS: {districts.crs.to_epsg()}")
print(f"Blocks: {len(blocks)} | CRS: {blocks.crs.to_epsg()}")

Precincts: 247 | CRS: 3081
Districts: 7 | CRS: 3081
Blocks: 16906 | CRS: 3081


In [2]:
# Step 1: Tag each census block with the precinct it falls in
blocks_with_precinct = gpd.sjoin(
    blocks, 
    precincts[['PCTKEY', 'geometry']], 
    how='left', 
    predicate='intersects'
)

print(f"Blocks after precinct join: {len(blocks_with_precinct)}")
print(f"Null precinct assignments: {blocks_with_precinct['PCTKEY'].isna().sum()}")
blocks_with_precinct[['SCTBKEY', 'total', 'PCTKEY']].head(5)

Blocks after precinct join: 24747
Null precinct assignments: 0


,SCTBKEY,total,PCTKEY
0,484530357003010,131,4530374
1,484530357003011,87,4530374
2,484530357003012,133,4530374
3,484530357003013,102,4530374
4,484530357003014,73,4530374


In [5]:
print(blocks_with_precinct.columns.tolist())

['BG', 'State', 'CNTY', 'TRT', 'BLK', 'SCTBKEY', 'vtd', 'blkkey', 'CTBKEY', 'CNTYKEY', 'Shape_area', 'Shape_len', 'total', 'anglo', 'asian', 'hisp', 'black', 'geometry', 'index_right', 'PCTKEY']


In [6]:
blocks_with_precinct = blocks_with_precinct.drop(columns=['index_right'])

# Step 2: Tag each census block with the district it falls in
blocks_with_both = gpd.sjoin(
    blocks_with_precinct,
    districts[['District', 'geometry']],
    how='left',
    predicate='intersects'
)

print(f"Blocks after district join: {len(blocks_with_both)}")
print(f"Null district assignments: {blocks_with_both['District'].isna().sum()}")
blocks_with_both[['SCTBKEY', 'total', 'PCTKEY', 'District']].head(5)

Blocks after district join: 27246
Null district assignments: 0


,SCTBKEY,total,PCTKEY,District
0,484530357003010,131,4530374,11
1,484530357003011,87,4530374,11
2,484530357003012,133,4530374,11
3,484530357003013,102,4530374,11
4,484530357003014,73,4530374,11


In [7]:
# Step 3: Group by (precinct, district) and sum population
fragments = blocks_with_both.groupby(['PCTKEY', 'District'])['total'].sum().reset_index()
fragments.columns = ['old_precinct_id', 'new_district_id', 'fragment_population']

# Step 4: Calculate weight = fragment_population / total precinct population
precinct_totals = fragments.groupby('old_precinct_id')['fragment_population'].sum()
fragments['precinct_total'] = fragments['old_precinct_id'].map(precinct_totals)
fragments['weight'] = fragments['fragment_population'] / fragments['precinct_total']

# Drop the helper column
fragments = fragments.drop(columns=['precinct_total'])

print(f"Total fragments: {len(fragments)}")
print(fragments.head(10))

Total fragments: 418
  old_precinct_id  new_district_id  fragment_population    weight
0         4530101               27                  139  0.006979
1         4530101               37                19778  0.993021
2         4530102               37                 5854  1.000000
3         4530103               37                 3743  1.000000
4         4530104               37                 5050  1.000000
5         4530105               10                 2369  0.068090
6         4530105               11                  228  0.006553
7         4530105               27                14260  0.409864
8         4530105               37                17935  0.515492
9         4530106               10                 3113  0.152748


In [8]:
# Validation: weights must sum to 1.0 per precinct
weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()

print(f"Min weight sum: {weight_sums.min():.6f}")
print(f"Max weight sum: {weight_sums.max():.6f}")
print(f"Any precinct not summing to 1.0: {(abs(weight_sums - 1.0) > 0.001).sum()}")

Min weight sum: 0.000000
Max weight sum: 1.000000
Any precinct not summing to 1.0: 2


In [9]:
# Find the problematic precincts
problem_precincts = weight_sums[abs(weight_sums - 1.0) > 0.001]
print("Problematic precincts:")
print(problem_precincts)

# Look at their fragments
print("\nTheir fragments:")
print(fragments[fragments['old_precinct_id'].isin(problem_precincts.index)])

Problematic precincts:
old_precinct_id
4530127    0.0
4530213    0.0
Name: weight, dtype: float64

Their fragments:
    old_precinct_id  new_district_id  fragment_population  weight
50          4530127               37                    0     NaN
110         4530213               10                    0     NaN
111         4530213               37                    0     NaN


In [10]:
# Fill NaN weights with 0 for zero-population precincts
fragments['weight'] = fragments['weight'].fillna(0)

# Re-run validation
weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()
print(f"Min weight sum: {weight_sums.min():.6f}")
print(f"Max weight sum: {weight_sums.max():.6f}")
print(f"Precincts not summing to 1.0 (excluding zero-pop): {(abs(weight_sums - 1.0) > 0.001).sum()}")

# Check for nulls
print(f"Null values in output: {fragments.isnull().sum().sum()}")

Min weight sum: 0.000000
Max weight sum: 1.000000
Precincts not summing to 1.0 (excluding zero-pop): 2
Null values in output: 0


In [11]:
# Improved validation - exclude zero population precincts
precinct_pops = fragments.groupby('old_precinct_id')['fragment_population'].sum()
zero_pop_precincts = precinct_pops[precinct_pops == 0].index

weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()
non_zero_weight_sums = weight_sums[~weight_sums.index.isin(zero_pop_precincts)]

print(f"Zero population precincts: {len(zero_pop_precincts)} (expected, no votes to allocate)")
print(f"Non-zero population precincts: {len(non_zero_weight_sums)}")
print(f"Min weight sum (non-zero precincts): {non_zero_weight_sums.min():.6f}")
print(f"Max weight sum (non-zero precincts): {non_zero_weight_sums.max():.6f}")
print(f"Any non-zero precinct not summing to 1.0: {(abs(non_zero_weight_sums - 1.0) > 0.001).sum()}")

Zero population precincts: 2 (expected, no votes to allocate)
Non-zero population precincts: 245
Min weight sum (non-zero precincts): 1.000000
Max weight sum (non-zero precincts): 1.000000
Any non-zero precinct not summing to 1.0: 0


In [12]:
# Save the population weights
fragments.to_csv('../data/processed/travis_population_weights.csv', index=False)

print("Saved to /data/processed/travis_population_weights.csv")
print(f"\nFinal output shape: {fragments.shape}")
print(f"Columns: {list(fragments.columns)}")
print(f"\nSample output:")
print(fragments.head(10))

Saved to /data/processed/travis_population_weights.csv

Final output shape: (418, 4)
Columns: ['old_precinct_id', 'new_district_id', 'fragment_population', 'weight']

Sample output:
  old_precinct_id  new_district_id  fragment_population    weight
0         4530101               27                  139  0.006979
1         4530101               37                19778  0.993021
2         4530102               37                 5854  1.000000
3         4530103               37                 3743  1.000000
4         4530104               37                 5050  1.000000
5         4530105               10                 2369  0.068090
6         4530105               11                  228  0.006553
7         4530105               27                14260  0.409864
8         4530105               37                17935  0.515492
9         4530106               10                 3113  0.152748
